In [1]:
import os
import re
from lxml import etree
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, CSV
import sparql_dataframe
import requests
import uuid
import time
from tqdm import tqdm
import os
import pickle
from collections import Counter
from shapely.wkt import loads
import pydeck as pdk

In [2]:
def extract_elements(xml_file):
    namespaces = {'tei': 'http://www.tei-c.org/ns/1.0'}
    with open(xml_file, 'rb') as file:
        tree = etree.parse(file)
    
    elements = {
        'placeName': tree.xpath('/tei:TEI/tei:text/tei:body/tei:sp/tei:ab/tei:seg/tei:reg/tei:placeName', namespaces=namespaces),
        'persName': tree.xpath('/tei:TEI/tei:text/tei:body/tei:sp/tei:ab/tei:seg/tei:reg/tei:persName', namespaces=namespaces),
        'target': tree.xpath('//tei:ptr/@target', namespaces=namespaces),
        'author': tree.xpath('//tei:author/tei:persName/text()', namespaces=namespaces),
        'title': tree.xpath('//tei:title/text()', namespaces=namespaces),
        'pubPlace': tree.xpath('//tei:pubPlace/text()', namespaces=namespaces),
        'date': tree.xpath('//tei:TEI/tei:teiHeader/tei:fileDesc/tei:sourceDesc/tei:bibl/tei:date/@when', namespaces=namespaces)
    }
    
    return tree, elements

In [3]:
def process_xml_folder(folder_path):
    data_place = []
    data_pers = []
    
    for filename in tqdm(os.listdir(folder_path), desc="Processing XML files"):
        if filename.endswith(".xml"):
            file_path = os.path.join(folder_path, filename)
            tree, elements = extract_elements(file_path)
            
            placeNames = [el.text for el in elements['placeName']]
            persNames = [el.text for el in elements['persName']]
            target = elements['target'][0] if elements['target'] else None
            author = elements['author'][0] if elements['author'] else None
            title = elements['title'][0] if elements['title'] else None
            pubPlace = elements['pubPlace'][0] if elements['pubPlace'] else None
            date = elements['date'][0] if elements['date'] else None
            
            placeName_counts = Counter(placeNames)
            persName_counts = Counter(persNames)
            
            for placeName in set(placeNames):
                data_place.append({
                    'target': target,
                    'title': title,
                    'pubPlace': pubPlace,
                    'date': date,
                    'author': author,
                    'place': placeName,
                    'placeName_num': placeName_counts[placeName]
                })
                
            for persName in set(persNames):
                data_pers.append({
                    'target': target,
                    'title': title,
                    'pubPlace': pubPlace,
                    'date': date,
                    'author': author,
                    'person': persName,
                    'persName_num': persName_counts[persName]
                })
    
    return data_place, data_pers

In [4]:
def create_dataframes(data_place, data_pers):
    df_place = pd.DataFrame(data_place)
    df_pers = pd.DataFrame(data_pers)
    
    df_place = df_place.drop_duplicates().sort_values(by='placeName_num', ascending=False)
    df_pers = df_pers.drop_duplicates().sort_values(by='persName_num', ascending=False)
    
    return df_place, df_pers

In [11]:
folder_path = '/Users/nicola/Documents/Academia/Projects/TextEnt/Processing/NER_test'

data_place, data_pers = process_xml_folder(folder_path)
df_place, df_pers = create_dataframes(data_place, data_pers)

Processing XML files: 100%|██████████| 4/4 [00:00<00:00, 151.80it/s]


In [12]:
df_place["place"] = df_place["place"].str.lower()
df_place.head(10)

,target,title,pubPlace,date,author,place,placeName_num
26,http://catalogue.bnf.fr/ark:/12148/cb45675147s,"La mort de Caton, ou L'illustre desespéré trag...",Grenoble,1648,"Auger, Jacques",rome,61
38,http://catalogue.bnf.fr/ark:/12148/cb313084529,"L'escolier de Salamanque, ou Les généreux enne...",Grenoble,1655,"Scarron, Paul",tolède,9
42,http://catalogue.bnf.fr/ark:/12148/cb393250582,"Faramond ou le Triomphe des Héros, tragicomédie",Grenoble,1672,Lapoujade,cologne,6
24,http://catalogue.bnf.fr/ark:/12148/cb45675147s,"La mort de Caton, ou L'illustre desespéré trag...",Grenoble,1648,"Auger, Jacques",pharsale,6
35,http://catalogue.bnf.fr/ark:/12148/cb313084529,"L'escolier de Salamanque, ou Les généreux enne...",Grenoble,1655,"Scarron, Paul",salamanque,4
15,http://catalogue.bnf.fr/ark:/12148/cb38760411p,L'essay des filles nouvelle comédie en trois a...,Grenoble,1699,None,paris,3
47,http://catalogue.bnf.fr/ark:/12148/cb393250582,"Faramond ou le Triomphe des Héros, tragicomédie",Grenoble,1672,Lapoujade,rome,2
44,http://catalogue.bnf.fr/ark:/12148/cb393250582,"Faramond ou le Triomphe des Héros, tragicomédie",Grenoble,1672,Lapoujade,france,2
17,http://catalogue.bnf.fr/ark:/12148/cb38760411p,L'essay des filles nouvelle comédie en trois a...,Grenoble,1699,None,piémont,2
16,http://catalogue.bnf.fr/ark:/12148/cb38760411p,L'essay des filles nouvelle comédie en trois a...,Grenoble,1699,None,flandre,2


In [13]:
df_pers["person"] = df_pers["person"].str.lower()
df_pers.head()

,target,title,pubPlace,date,author,person,persName_num
156,http://catalogue.bnf.fr/ark:/12148/cb45675147s,"La mort de Caton, ou L'illustre desespéré trag...",Grenoble,1648,"Auger, Jacques",césar,85
303,http://catalogue.bnf.fr/ark:/12148/cb313084529,"L'escolier de Salamanque, ou Les généreux enne...",Grenoble,1655,"Scarron, Paul",crispin,77
505,http://catalogue.bnf.fr/ark:/12148/cb393250582,"Faramond ou le Triomphe des Héros, tragicomédie",Grenoble,1672,Lapoujade,balamir,68
52,http://catalogue.bnf.fr/ark:/12148/cb38760411p,L'essay des filles nouvelle comédie en trois a...,Grenoble,1699,None,pascariel,66
464,http://catalogue.bnf.fr/ark:/12148/cb393250582,"Faramond ou le Triomphe des Héros, tragicomédie",Grenoble,1672,Lapoujade,faramond,63


In [14]:
df_place_filter = df_place.groupby('target', group_keys=False).apply(lambda x: x.nlargest(2, 'placeName_num'))

/var/folders/y2/1xj8qbr56dvf504d6g874cp00000gn/T/ipykernel_44400/2248687726.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_place_filter = df_place.groupby('target', group_keys=False).apply(lambda x: x.nlargest(2, 'placeName_num'))


In [15]:
df_place_filter['uuid'] = df_place_filter['place'].apply(lambda x: str(uuid.uuid5(uuid.NAMESPACE_DNS, x)))

In [16]:
df_place_filter

,target,title,pubPlace,date,author,place,placeName_num,uuid
38,http://catalogue.bnf.fr/ark:/12148/cb313084529,"L'escolier de Salamanque, ou Les généreux enne...",Grenoble,1655,"Scarron, Paul",tolède,9,962852b2-97e0-558b-b021-2148ee1823c1
35,http://catalogue.bnf.fr/ark:/12148/cb313084529,"L'escolier de Salamanque, ou Les généreux enne...",Grenoble,1655,"Scarron, Paul",salamanque,4,7e1fedfe-f21f-52bf-a07f-23a331c32c13
15,http://catalogue.bnf.fr/ark:/12148/cb38760411p,L'essay des filles nouvelle comédie en trois a...,Grenoble,1699,None,paris,3,12c92cce-a675-51df-9da5-c91db5c631dd
17,http://catalogue.bnf.fr/ark:/12148/cb38760411p,L'essay des filles nouvelle comédie en trois a...,Grenoble,1699,None,piémont,2,ebf0c64c-09ac-59a0-b2db-ef33e270a247
42,http://catalogue.bnf.fr/ark:/12148/cb393250582,"Faramond ou le Triomphe des Héros, tragicomédie",Grenoble,1672,Lapoujade,cologne,6,19da8780-7362-5463-b18e-b35744f1d0ce
47,http://catalogue.bnf.fr/ark:/12148/cb393250582,"Faramond ou le Triomphe des Héros, tragicomédie",Grenoble,1672,Lapoujade,rome,2,cbef6bfb-ce53-50a5-a154-fb3e13aa618e
26,http://catalogue.bnf.fr/ark:/12148/cb45675147s,"La mort de Caton, ou L'illustre desespéré trag...",Grenoble,1648,"Auger, Jacques",rome,61,cbef6bfb-ce53-50a5-a154-fb3e13aa618e
24,http://catalogue.bnf.fr/ark:/12148/cb45675147s,"La mort de Caton, ou L'illustre desespéré trag...",Grenoble,1648,"Auger, Jacques",pharsale,6,d8437283-72b8-5cc0-8aea-7bf5ff0328b8


In [ ]:
#df_place.to_csv('/Users/carboni/Documents/Academia/Projects/TextEnt/output/df_place.csv', index=False)
#df_pers.to_csv('/Users/carboni/Documents/Academia/Projects/TextEnt/output/df_pers.csv', index=False)

### Removing Articles

In [17]:
articles = [
    "l'",
    "le ",
    "la ",
    "les ",
    "un ",
    "une ",
    "des ",
    "du ",
    "de la ",
    "de l'",
    "de ",
    "d'",
    "au ",
    "aux ",
    "à ",
    "chez ",
    "sur ",
    "en ",
    "dans "
]

In [18]:
pattern = r'^(' + '|'.join([re.escape(article) for article in articles]) + r')'
df_place_filter['place_no_article'] = df_place_filter['place'].str.replace(pattern, '', flags=re.IGNORECASE)

## Wikidata Query

In [19]:
def save_progress(results, filename='sparql_results_temp.csv'):
    temp_df = pd.DataFrame(results)
    temp_df.to_csv(filename, index=False)

def load_progress(filename='sparql_results_temp.csv'):
    if os.path.exists(filename):
        temp_df = pd.read_csv(filename)
        return temp_df.to_dict('records')
    return []

def load_query_cache(filename='query_cache.pkl'):
    if os.path.exists(filename):
        with open(filename, 'rb') as f:
            return pickle.load(f)
    return {}

def save_query_cache(cache, filename='query_cache.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(cache, f)

In [31]:
def query_wikidata(place):
    if place in query_cache:
        return query_cache[place]

    query_template = """
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX wikibase: <http://wikiba.se/ontology#>
    PREFIX schema: <http://schema.org/>

    SELECT ?a ?name ?coordinates ?typeLabel ?culture ?start_time ?end_time ?country (group_concat(?countryLabel; separator=";; ") AS ?countryLabels) ?osm_id ?sitelinks ?roman_atlas_id ?pleiades_id ?topostext_id ?myths_id ?poleis_id ?manto_id   
    WHERE {{
      {{
        SELECT ?a (MAX(?sitelinks) AS ?maxSitelinks)
        WHERE {{
          VALUES ?category {{ wd:Q6256 wd:Q82794 wd:Q1620908 }}
          ?a (wdt:P31)/((wdt:P279)*) ?category.
          ?a rdfs:label ?name .
          ?a ^schema:about/wikibase:sitelinks ?sitelinks .

          FILTER (LANG(?name) = "fr") .
          FILTER REGEX(STR(?name), "^{place}$", "i") .
        }}
        GROUP BY ?a
        ORDER BY DESC(?maxSitelinks)
        LIMIT 1
      }}

      ?a rdfs:label ?name .
      ?a wdt:P31 ?type .
      ?a wdt:P625 ?coordinates . 
      ?a ^schema:about/wikibase:sitelinks ?sitelinks .
     
      ?type rdfs:label ?typeLabel .
      
      
      OPTIONAL {{
        ?a wdt:P17 ?country .
        ?country rdfs:label ?countryLabel .
        FILTER (LANG(?countryLabel) = "en") .
      }}
      
    OPTIONAL {{
         ?a wdt:P2596 ?culture . 
         ?culture wdt:P580 ?start_time .
         ?culture wdt:P582 ?end_time .
      }} 
      
    OPTIONAL {{
        ?a wdt:P1584 ?pleiades_id
      }}

     OPTIONAL {{
        ?a wdt:P402 ?osm_id
      }}
      
    OPTIONAL {{
        ?a wdt:P8068 ?topostext_id . 
      }}
    OPTIONAL {{
        ?a wdt:P361 ?partOf
      }}
    OPTIONAL {{
        ?a wdt:P1936 ?roman_atlas_id . 
      }}  
    OPTIONAL {{
        ?a wdt:P12402 ?myths_id . 
      }}  
    OPTIONAL {{
        ?a wdt:P8137 ?poleis_id .
      }}
    OPTIONAL {{
        ?a wdt:P9736 ?manto_id .
      }} 

      FILTER (LANG(?name) = "en") .
      FILTER (LANG(?typeLabel) = "en") .
    }}
    GROUP BY ?a ?name ?coordinates ?typeLabel ?culture ?start_time ?end_time ?country ?sitelinks ?osm_id ?roman_atlas_id ?pleiades_id ?topostext_id ?myths_id ?poleis_id ?manto_id
    ORDER BY DESC(?sitelinks)
    """
    
    query = query_template.format(place=place)
    #url = 'https://qlever.cs.uni-freiburg.de/api/wikidata'
    url = 'http://10.194.68.72:7001' #internal unige
    response = requests.get(url, params={'query': query, 'output': 'json'})
    
    if response.status_code == 200:
        raw_results = response.json().get('results', {}).get('bindings', [])
        # Process results to ensure all values are strings
        processed_results = []
        for result in raw_results:
            processed_result = {key: str(value.get('value', '')) for key, value in result.items()}
            processed_results.append(processed_result)
        query_cache[place] = processed_results
        return processed_results
    else:
        query_cache[place] = None  # Cache failed queries as well
        return None

In [32]:
# Initialize or load query cache
query_cache = load_query_cache()

# Load existing results if any
results = load_progress()

# Get the set of already processed places
processed_places = set([entry['place'] for entry in results])

# Extract unique places from your DataFrame
unique_places_df = df_place_filter.drop_duplicates(subset='place')

# Initialize a counter for saving progress periodically
counter = 0
save_every_n = 10  # Save progress every N iterations

# Iterate over unique places
for index, row in tqdm(unique_places_df.iterrows(), total=len(unique_places_df), desc="Querying QLever"):
    place = row['place']
    target = row['target']
    uuid = row['uuid']
    
    if place in processed_places:
        continue
    
    sparql_results = query_wikidata(place)
    
    if sparql_results:
        for result in sparql_results:
            results.append({
                'place': place,
                'target': target,
                'uuid': uuid,
                'wikidata_id': result.get('a', ''),
                'name': result.get('name', ''),
                'coordinates': result.get('coordinates', ''),
                'typeLabel': result.get('typeLabel', ''),
                'country': result.get('country', ''),
                'countryLabel': result.get('countryLabels', ''),
                'culture': result.get('culture', ''),
                'start_time': result.get('start_time', ''),
                'end_time': result.get('end_time', ''),
                'partOf': result.get('partOf', ''),
                'sitelinks': result.get('sitelinks', ''),
                'osm_id': result.get('osm_id', ''),
                'roman_atlas_id': result.get('roman_atlas_id', ''),
                'pleiades_id': result.get('pleiades_id', ''),
                'topostext_id': result.get('topostext_id', ''),
                'myths_id': result.get('myths_id', ''),
                'poleis_id': result.get('poleis_id', ''),
                'manto_id': result.get('manto_id', '')
            })
    else:
        # Handle cases where there is no result
        results.append({
            'place': place,
            'uuid': uuid,
            'target': target,
            'wikidata_id': '',
            'name': '',
            'coordinates': '',
            'typeLabel': '',
            'country': '',
            'countryLabels': '',
            'culture': '',
            'start_time': '',
            'end_time': '',
            'partOf': '',
            'sitelinks': '',
            'osm_id': '',
            'roman_atlas_id': '',
            'pleiades_id': '',
            'topostext_id': '',
            'myths_id': '',
            'poleis_id': '',
            'manto_id': ''
        })
    
    processed_places.add(place)
    counter += 1
    
    # Save progress and cache every N iterations
    if counter % save_every_n == 0:
        save_progress(results)
        save_query_cache(query_cache)
        # Optionally, print a message
        # print(f"Saved progress after processing {counter} places.")
    
    # Optional: Remove or adjust sleep time if necessary
    time.sleep(1)

# After processing all places, save final progress and cache
save_progress(results)
save_query_cache(query_cache)

# Convert the results to a new DataFrame
df_place_coordinates = pd.DataFrame(results)

Querying QLever: 100%|██████████| 7/7 [00:18<00:00,  2.62s/it]


In [33]:
!rm query_cache.pkl
!rm sparql_results_temp.csv

In [34]:
df_place_coordinates

,place,target,uuid,wikidata_id,name,coordinates,typeLabel,country,countryLabel,culture,...,end_time,partOf,sitelinks,osm_id,roman_atlas_id,pleiades_id,topostext_id,myths_id,poleis_id,manto_id
0,tolède,http://catalogue.bnf.fr/ark:/12148/cb313084529,962852b2-97e0-558b-b021-2148ee1823c1,http://www.wikidata.org/entity/Q5836,Toledo,POINT(-4.033333 39.866667),municipality of Spain,http://www.wikidata.org/entity/Q29,Spain,,...,,,126,342999,,266066,,,,
1,salamanque,http://catalogue.bnf.fr/ark:/12148/cb313084529,7e1fedfe-f21f-52bf-a07f-23a331c32c13,http://www.wikidata.org/entity/Q15695,Salamanca,POINT(-5.664167 40.965000),municipality of Spain,http://www.wikidata.org/entity/Q29,Spain,,...,,,113,344342,16865,236642,410000USal,,,
2,paris,http://catalogue.bnf.fr/ark:/12148/cb38760411p,12c92cce-a675-51df-9da5-c91db5c631dd,http://www.wikidata.org/entity/Q90,Paris,POINT(2.352222 48.856667),department of France,http://www.wikidata.org/entity/Q142,France,,...,,,348,7444,,,,,,
3,paris,http://catalogue.bnf.fr/ark:/12148/cb38760411p,12c92cce-a675-51df-9da5-c91db5c631dd,http://www.wikidata.org/entity/Q90,Paris,POINT(2.352222 48.856667),global city,http://www.wikidata.org/entity/Q142,France,,...,,,348,7444,,,,,,
4,paris,http://catalogue.bnf.fr/ark:/12148/cb38760411p,12c92cce-a675-51df-9da5-c91db5c631dd,http://www.wikidata.org/entity/Q90,Paris,POINT(2.352222 48.856667),largest city,http://www.wikidata.org/entity/Q142,France,,...,,,348,7444,,,,,,
5,paris,http://catalogue.bnf.fr/ark:/12148/cb38760411p,12c92cce-a675-51df-9da5-c91db5c631dd,http://www.wikidata.org/entity/Q90,Paris,POINT(2.352222 48.856667),megacity,http://www.wikidata.org/entity/Q142,France,,...,,,348,7444,,,,,,
6,paris,http://catalogue.bnf.fr/ark:/12148/cb38760411p,12c92cce-a675-51df-9da5-c91db5c631dd,http://www.wikidata.org/entity/Q90,Paris,POINT(2.352222 48.856667),metropolis,http://www.wikidata.org/entity/Q142,France,,...,,,348,7444,,,,,,
7,paris,http://catalogue.bnf.fr/ark:/12148/cb38760411p,12c92cce-a675-51df-9da5-c91db5c631dd,http://www.wikidata.org/entity/Q90,Paris,POINT(2.352222 48.856667),territorial collectivity of France with specia...,http://www.wikidata.org/entity/Q142,France,,...,,,348,7444,,,,,,
8,piémont,http://catalogue.bnf.fr/ark:/12148/cb38760411p,ebf0c64c-09ac-59a0-b2db-ef33e270a247,http://www.wikidata.org/entity/Q1216,Piedmont,POINT(7.916667 45.250000),region of Italy,http://www.wikidata.org/entity/Q38,Italy,,...,,,152,44874,,,,,,
9,cologne,http://catalogue.bnf.fr/ark:/12148/cb393250582,19da8780-7362-5463-b18e-b35744f1d0ce,http://www.wikidata.org/entity/Q365,Cologne,POINT(6.957778 50.942222),big city,http://www.wikidata.org/entity/Q183,Germany,,...,,,184,62578,299,108751,509070UCol,,,


aggregate instead of using SPARQL concat

In [39]:
concat_cols = ['typeLabel', 'countryLabel']

In [40]:
def get_agg_func(col):
    if col in concat_cols:
        return lambda x: ';;'.join(x.unique())
    else:
        return 'first'

In [41]:
agg_dict = {col: get_agg_func(col) for col in df_place_coordinates.columns if col != 'uuid'}

In [42]:
df_place_concat = df_place_coordinates.groupby('uuid', as_index=False).agg(agg_dict)

In [43]:
df_place_concat

,uuid,place,target,wikidata_id,name,coordinates,typeLabel,country,countryLabel,culture,...,end_time,partOf,sitelinks,osm_id,roman_atlas_id,pleiades_id,topostext_id,myths_id,poleis_id,manto_id
0,12c92cce-a675-51df-9da5-c91db5c631dd,paris,http://catalogue.bnf.fr/ark:/12148/cb38760411p,http://www.wikidata.org/entity/Q90,Paris,POINT(2.352222 48.856667),department of France;;global city;;largest cit...,http://www.wikidata.org/entity/Q142,France,,...,,,348,7444,,,,,,
1,19da8780-7362-5463-b18e-b35744f1d0ce,cologne,http://catalogue.bnf.fr/ark:/12148/cb393250582,http://www.wikidata.org/entity/Q365,Cologne,POINT(6.957778 50.942222),big city;;Hanseatic city;;metropolis;;Roman ci...,http://www.wikidata.org/entity/Q183,Germany,,...,,,184,62578,299,108751,509070UCol,,,
2,7e1fedfe-f21f-52bf-a07f-23a331c32c13,salamanque,http://catalogue.bnf.fr/ark:/12148/cb313084529,http://www.wikidata.org/entity/Q15695,Salamanca,POINT(-5.664167 40.965000),municipality of Spain,http://www.wikidata.org/entity/Q29,Spain,,...,,,113,344342,16865,236642,410000USal,,,
3,962852b2-97e0-558b-b021-2148ee1823c1,tolède,http://catalogue.bnf.fr/ark:/12148/cb313084529,http://www.wikidata.org/entity/Q5836,Toledo,POINT(-4.033333 39.866667),municipality of Spain,http://www.wikidata.org/entity/Q29,Spain,,...,,,126,342999,,266066,,,,
4,cbef6bfb-ce53-50a5-a154-fb3e13aa618e,rome,http://catalogue.bnf.fr/ark:/12148/cb393250582,http://www.wikidata.org/entity/Q220,Rome,POINT(12.482778 41.893056),abolished municipality in Italy;;big city;;bor...,http://www.wikidata.org/entity/Q38,Italy,,...,,,329,41485,1438,423025,,ROME2,,
5,d8437283-72b8-5cc0-8aea-7bf5ff0328b8,pharsale,http://catalogue.bnf.fr/ark:/12148/cb45675147s,http://www.wikidata.org/entity/Q985596,Farsala,POINT(22.383333 39.283333),town,http://www.wikidata.org/entity/Q41,Greece,,...,,,36,,21980,,393224UPha,,413,
6,ebf0c64c-09ac-59a0-b2db-ef33e270a247,piémont,http://catalogue.bnf.fr/ark:/12148/cb38760411p,http://www.wikidata.org/entity/Q1216,Piedmont,POINT(7.916667 45.250000),region of Italy,http://www.wikidata.org/entity/Q38,Italy,,...,,,152,44874,,,,,,


In [24]:
df_place_concat.to_csv('df_place_coordinates.csv', index=False)

### Filter labelType

In [54]:
filter_types = ['river', 'lake', 'region']

In [55]:
mask = df_place_concat['typeLabel'].str.contains('|'.join(filter_types), case=False, na=False)

In [56]:
mask = mask & df_place_concat['osm_id'].notnull()
df_filtered = df_place_concat[mask].copy()

In [61]:
def get_osm_wkt(osm_id):
    # Ensure osm_id is a string (in case it's not)
    osm_id_str = str(osm_id).strip()
    
    # Build the SPARQL query, substituting the osm_id
    query_template = """
    PREFIX ogc: <http://www.opengis.net/rdf#>
    PREFIX osmrel: <https://www.openstreetmap.org/relation/>
    PREFIX geo: <http://www.opengis.net/ont/geosparql#>
    PREFIX osm: <https://www.openstreetmap.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX osmkey: <https://www.openstreetmap.org/wiki/Key:>
    SELECT ?shape WHERE {{
      osmrel:{osm_id} geo:hasGeometry/geo:asWKT ?shape
    }}
    """
    query = query_template.format(osm_id=osm_id_str)
    url = 'https://qlever.cs.uni-freiburg.de/api/osm-planet'

    # Send the request
    try:
        response = requests.get(url, params={'query': query, 'output': 'json'})
        if response.status_code == 200:
            results = response.json().get('results', {}).get('bindings', [])
            if results:
                # Get the 'shape' value
                shape = results[0].get('shape', {}).get('value', '')
                return shape
            else:
                # No results found
                return None
        else:
            print(f"Error querying osm_id {osm_id}: HTTP {response.status_code}")
            return None
    except Exception as e:
        print(f"Exception querying osm_id {osm_id}: {e}")
        return None

In [62]:
def fetch_osm_wkt(row):
    return get_osm_wkt(row['osm_id'])

In [63]:
df_filtered['osm_wkt'] = df_filtered.apply(fetch_osm_wkt, axis=1)

In [64]:
df_filtered

,uuid,place,target,wikidata_id,name,coordinates,typeLabel,country,countryLabel,culture,...,partOf,sitelinks,osm_id,roman_atlas_id,pleiades_id,topostext_id,myths_id,poleis_id,manto_id,osm_wkt
6,ebf0c64c-09ac-59a0-b2db-ef33e270a247,piémont,http://catalogue.bnf.fr/ark:/12148/cb38760411p,http://www.wikidata.org/entity/Q1216,Piedmont,POINT(7.916667 45.250000),region of Italy,http://www.wikidata.org/entity/Q38,Italy,,...,,152,44874,,,,,,,"MULTIPOLYGON(((6.6272658 45.1068023,6.6272839 ..."


In [65]:
df_place_concat['osm_wkt'] = None
df_place_concat.loc[df_filtered.index, 'osm_wkt'] = df_filtered['osm_wkt']

In [66]:
df_place_concat

,uuid,place,target,wikidata_id,name,coordinates,typeLabel,country,countryLabel,culture,...,partOf,sitelinks,osm_id,roman_atlas_id,pleiades_id,topostext_id,myths_id,poleis_id,manto_id,osm_wkt
0,12c92cce-a675-51df-9da5-c91db5c631dd,paris,http://catalogue.bnf.fr/ark:/12148/cb38760411p,http://www.wikidata.org/entity/Q90,Paris,POINT(2.352222 48.856667),department of France;;global city;;largest cit...,http://www.wikidata.org/entity/Q142,France,,...,,348,7444,,,,,,,None
1,19da8780-7362-5463-b18e-b35744f1d0ce,cologne,http://catalogue.bnf.fr/ark:/12148/cb393250582,http://www.wikidata.org/entity/Q365,Cologne,POINT(6.957778 50.942222),big city;;Hanseatic city;;metropolis;;Roman ci...,http://www.wikidata.org/entity/Q183,Germany,,...,,184,62578,299,108751,509070UCol,,,,None
2,7e1fedfe-f21f-52bf-a07f-23a331c32c13,salamanque,http://catalogue.bnf.fr/ark:/12148/cb313084529,http://www.wikidata.org/entity/Q15695,Salamanca,POINT(-5.664167 40.965000),municipality of Spain,http://www.wikidata.org/entity/Q29,Spain,,...,,113,344342,16865,236642,410000USal,,,,None
3,962852b2-97e0-558b-b021-2148ee1823c1,tolède,http://catalogue.bnf.fr/ark:/12148/cb313084529,http://www.wikidata.org/entity/Q5836,Toledo,POINT(-4.033333 39.866667),municipality of Spain,http://www.wikidata.org/entity/Q29,Spain,,...,,126,342999,,266066,,,,,None
4,cbef6bfb-ce53-50a5-a154-fb3e13aa618e,rome,http://catalogue.bnf.fr/ark:/12148/cb393250582,http://www.wikidata.org/entity/Q220,Rome,POINT(12.482778 41.893056),abolished municipality in Italy;;big city;;bor...,http://www.wikidata.org/entity/Q38,Italy,,...,,329,41485,1438,423025,,ROME2,,,None
5,d8437283-72b8-5cc0-8aea-7bf5ff0328b8,pharsale,http://catalogue.bnf.fr/ark:/12148/cb45675147s,http://www.wikidata.org/entity/Q985596,Farsala,POINT(22.383333 39.283333),town,http://www.wikidata.org/entity/Q41,Greece,,...,,36,,21980,,393224UPha,,413,,None
6,ebf0c64c-09ac-59a0-b2db-ef33e270a247,piémont,http://catalogue.bnf.fr/ark:/12148/cb38760411p,http://www.wikidata.org/entity/Q1216,Piedmont,POINT(7.916667 45.250000),region of Italy,http://www.wikidata.org/entity/Q38,Italy,,...,,152,44874,,,,,,,"MULTIPOLYGON(((6.6272658 45.1068023,6.6272839 ..."


#### Further Refiniment

if it is a river or a waterway, get the coordinate from OpenStreetMap. Possible to do using the instance in QLever (sparql = https://qlever.cs.uni-freiburg.de/api/osm-planet
)
```
PREFIX ogc: <http://www.opengis.net/rdf#>
PREFIX osmrel: <https://www.openstreetmap.org/relation/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX osm: <https://www.openstreetmap.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX osmkey: <https://www.openstreetmap.org/wiki/Key:>
SELECT * WHERE {
  osmrel:2188548 geo:hasGeometry/geo:asWKT ?shape
}
```
problems: somehow throgh python it does not retrieve the type

if results are empty, query TGN. Althought it is quite slow:

```
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?place ?geojson ?typeLabel WHERE {
  ?place a crm:E53_Place .
  ?place rdfs:label "oristano" .
  ?place crm:P1_is_identified_by ?geometry .
  ?geometry crm:P2_has_type <http://geojson.org> ;
            crm:P90_has_value ?geojson .
  ?place crm:P2_has_type ?type .
  ?type rdfs:label ?typeLabel .
} LIMIT 100
```

Other option, use world historical gazetteer API
https://whgazetteer.org/api/index/?name=oristano

## Map Creation

In [ ]:
df_place_coordinates = df_place_coordinates.dropna(subset=['coordinates'])

In [ ]:
df_place_coordinates['place_number'] = df_place_coordinates.groupby(['place', 'country'])['place'].transform('count')

In [ ]:
df_place_coordinates['geometry'] = df_place_coordinates['coordinates'].apply(loads)

In [ ]:
df_place_coordinates['latitude'] = df_place_coordinates['geometry'].apply(lambda geom: geom.y)
df_place_coordinates['longitude'] = df_place_coordinates['geometry'].apply(lambda geom: geom.x)

In [ ]:
df_no_duplicates = df_place_coordinates.drop_duplicates(subset=['place', 'country'])

In [ ]:
df_no_duplicates

In [ ]:
df_no_duplicates = df_no_duplicates.drop(columns=['geometry'])

In [ ]:
scatter_layer = pdk.Layer(
    'ScatterplotLayer',
    df_no_duplicates,
    opacity=0.6,
    get_position=['longitude', 'latitude'],
    get_radius='place_number * 5000',
    get_fill_color=[255, 0, 0],  # Red 
    pickable=True,
    stroked=True,
    get_line_color=[255,255,255]
)

In [ ]:
view_state = pdk.ViewState(
    latitude=df_no_duplicates['latitude'].mean(),
    longitude=df_no_duplicates['longitude'].mean(),
    zoom=3,
)

In [ ]:
tooltip = {
    "html": "<b>{place_number}</b> reference to <b>{place}</b>",
    "style": {"background": "grey", "color": "white", "font-family": '"Helvetica Neue", Arial', "z-index": "10000"},
}

In [ ]:
deck = pdk.Deck(
    layers=[scatter_layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_provider="carto",
    map_style="light" #possible here to go for light’, ‘dark’, ‘road’, ‘satellite’, 
    #‘dark_no_labels’, and ‘light_no_labels’. Also possible to use mapbox. To change together with the 
    #parameters on scatter_layer (e.g. opacity!)
)

In [ ]:
deck.to_html(filename='textent_map.html', offline=True, open_browser=False, notebook_display=False)